# Hindi-English Machine Translation (A3 Project)

**Student**: Htut Ko Ko  
**Course**: Natural Language Understanding  
**Task**: Hindi (hi) <-> English (en) Translation using Transformer

## Project Overview
This notebook implements a Neural Machine Translation system using a **Transformer** architecture.
We use the **Opus-100** dataset for Hindi-English parallel data (since ALT does not cover Hindi).
We use **SentencePiece** for subword tokenization.


In [1]:
import os
import math
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
import sentencepiece as spm

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

Using device: cuda


## 2. Data Loading (Opus-100)
Loading Hindi-English pairs from Opus-100.

In [2]:
print("Loading Opus-100 Dataset (Hindi-English)...")
try:
    # Opus-100 has 'en-hi'
    dataset = load_dataset("opus100", "en-hi", split="train+validation+test")
    print(f"Loaded {len(dataset)} sentences from Opus-100 dataset.")

    # Extract data
    data = []
    for item in dataset:
        if 'translation' in item:
            if 'hi' in item['translation'] and 'en' in item['translation']:
                data.append({
                    'hi': item['translation']['hi'],
                    'en': item['translation']['en']
                })

    # Limit to manageable size for training on Colab if too large (Opus is huge)
    if len(data) > 50000:
        import random
        random.shuffle(data)
        data = data[:50000]
        print("Subsampled to 50,000 examples for efficiency.")

    print(f"Extracted {len(data)} Hindi-English pairs.")

except Exception as e:
    print(f"Error loading from HF: {e}")

Loading Opus-100 Dataset (Hindi-English)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

en-hi/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

en-hi/train-00000-of-00001.parquet:   0%|          | 0.00/65.2M [00:00<?, ?B/s]

en-hi/validation-00000-of-00001.parquet:   0%|          | 0.00/247k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/534319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Loaded 538319 sentences from Opus-100 dataset.
Subsampled to 50,000 examples for efficiency.
Extracted 50000 Hindi-English pairs.


In [3]:
df = pd.DataFrame(data)
print(df.head())

df = df.dropna(subset=['hi', 'en'])
df['hi'] = df['hi'].astype(str)
df['en'] = df['en'].astype(str)
df = df[df['hi'].str.strip() != '']
df = df[df['en'].str.strip() != '']

                                                  hi  \
0                                             - ल ..   
1  अलिफ़॰ लाम॰ रा॰। यह एक किताब है जिसकी आयतें पक...   
2                          इन बेबी के बिना कैसे रहे।   
3  वाहीआवाCity name (optional, probably does not ...   
4           - ट्रेवर 'Atlantis.u के की uLost शहर: मम   

                                                  en  
0                                             - L...  
1  Alif Lam Ra (This is) a Book, whose verses are...  
2  Pre-ordering a prossie to murder at the next t...  
3                                            Wahiawa  
4                -'uLost City of Atlantis.u' TREVOR:  


## 3. Tokenization

In [4]:
# Save texts to files
with open('train_hi.txt', 'w', encoding='utf-8') as f:
    for line in df['hi']: f.write(line + '\n')

with open('train_en_hi.txt', 'w', encoding='utf-8') as f:
    for line in df['en']: f.write(line + '\n')

# Train SentencePiece models
vocab_size = 8000 # Increased for larger dataset/diversity
model_type = 'bpe'

print("Training Hindi Tokenizer...")
spm.SentencePieceTrainer.train(
    input='train_hi.txt',
    model_prefix='spm_hi',
    vocab_size=vocab_size,
    model_type=model_type,
    pad_id=0, bos_id=1, eos_id=2, unk_id=3
)

print("Training English Tokenizer (for Hindi pair)...")
spm.SentencePieceTrainer.train(
    input='train_en_hi.txt',
    model_prefix='spm_en_hi',
    vocab_size=vocab_size,
    model_type=model_type,
    pad_id=0, bos_id=1, eos_id=2, unk_id=3
)

sp_src = spm.SentencePieceProcessor(model_file='spm_hi.model')
sp_trg = spm.SentencePieceProcessor(model_file='spm_en_hi.model')

Training Hindi Tokenizer...
Training English Tokenizer (for Hindi pair)...


## 4. Dataset & Model

In [5]:
class TranslationDataset(Dataset):
    def __init__(self, df, sp_src, sp_trg):
        self.data = df
        self.sp_src = sp_src
        self.sp_trg = sp_trg

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        src_text = self.data.iloc[idx]['hi']
        trg_text = self.data.iloc[idx]['en']
        src_ids = [self.sp_src.bos_id()] + self.sp_src.encode(src_text, out_type=int) + [self.sp_src.eos_id()]
        trg_ids = [self.sp_trg.bos_id()] + self.sp_trg.encode(trg_text, out_type=int) + [self.sp_trg.eos_id()]
        return torch.tensor(src_ids), torch.tensor(trg_ids)

def collate_fn(batch):
    src_batch, trg_batch = [], []
    for src, trg in batch:
        src_batch.append(src)
        trg_batch.append(trg)
    src_pad = pad_sequence(src_batch, batch_first=True, padding_value=0)
    trg_pad = pad_sequence(trg_batch, batch_first=True, padding_value=0)
    return src_pad, trg_pad

train_dataset = TranslationDataset(df, sp_src, sp_trg)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)

In [6]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(1), :]
        return self.dropout(x)

class TransformerModel(nn.Module):
    def __init__(self, src_vocab_size, trg_vocab_size,
                 d_model=256, nhead=4, num_encoder_layers=2,
                 num_decoder_layers=2, dim_feedforward=512, dropout=0.1, pad_idx=0):
        super(TransformerModel, self).__init__()
        self.d_model = d_model
        self.pad_idx = pad_idx
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.trg_embedding = nn.Embedding(trg_vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        self.transformer = nn.Transformer(d_model=d_model, nhead=nhead, num_encoder_layers=num_encoder_layers, num_decoder_layers=num_decoder_layers, dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True)
        self.fc_out = nn.Linear(d_model, trg_vocab_size)

    def forward(self, src, trg):
        src_key_padding_mask = (src == self.pad_idx)
        trg_mask = self.transformer.generate_square_subsequent_mask(trg.size(1)).to(src.device)
        src_emb = self.pos_encoder(self.src_embedding(src) * math.sqrt(self.d_model))
        trg_emb = self.pos_encoder(self.trg_embedding(trg) * math.sqrt(self.d_model))
        output = self.transformer(src=src_emb, tgt=trg_emb, tgt_mask=trg_mask, src_key_padding_mask=src_key_padding_mask)
        return self.fc_out(output)

In [7]:
model = TransformerModel(vocab_size, vocab_size).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0005)
criterion = nn.CrossEntropyLoss(ignore_index=0)

print("Starting Training...")
for epoch in range(10): # 10 Epochs for demo
    model.train()
    epoch_loss = 0
    for i, (src, trg) in enumerate(train_loader):
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()
        output = model(src, trg[:, :-1])
        output = output.contiguous().view(-1, output.shape[-1])
        trg = trg[:, 1:].contiguous().view(-1)
        loss = criterion(output, trg)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        if i % 100 == 0: print(f"Step {i}, Loss: {loss.item():.3f}")
    print(f"Epoch {epoch+1} Loss: {epoch_loss/len(train_loader):.3f}")

    # Save
    torch.save(model.state_dict(), 'transformer_model_hi.pt')

Starting Training...
Step 0, Loss: 9.189
Step 100, Loss: 6.300
Step 200, Loss: 5.875
Step 300, Loss: 5.781
Step 400, Loss: 5.471
Step 500, Loss: 5.690
Step 600, Loss: 5.303
Step 700, Loss: 5.245
Epoch 1 Loss: 5.626
Step 0, Loss: 4.651
Step 100, Loss: 4.864
Step 200, Loss: 4.755
Step 300, Loss: 4.875
Step 400, Loss: 4.955
Step 500, Loss: 4.613
Step 600, Loss: 4.793
Step 700, Loss: 4.608
Epoch 2 Loss: 4.749
Step 0, Loss: 4.476
Step 100, Loss: 4.460
Step 200, Loss: 4.573
Step 300, Loss: 4.310
Step 400, Loss: 4.169
Step 500, Loss: 4.296
Step 600, Loss: 4.434
Step 700, Loss: 4.135
Epoch 3 Loss: 4.344
Step 0, Loss: 4.275
Step 100, Loss: 4.253
Step 200, Loss: 4.010
Step 300, Loss: 4.083
Step 400, Loss: 4.119
Step 500, Loss: 3.813
Step 600, Loss: 3.818
Step 700, Loss: 4.239
Epoch 4 Loss: 4.054
Step 0, Loss: 3.653
Step 100, Loss: 3.975
Step 200, Loss: 3.831
Step 300, Loss: 3.881
Step 400, Loss: 3.946
Step 500, Loss: 3.686
Step 600, Loss: 3.760
Step 700, Loss: 3.940
Epoch 5 Loss: 3.835
Step 0, L

In [8]:
# Copy to app
import shutil
os.makedirs('app/models', exist_ok=True)
shutil.copy('transformer_model_hi.pt', 'app/models/transformer_model_hi.pt')
shutil.copy('spm_hi.model', 'app/models/spm_hi.model')
shutil.copy('spm_en_hi.model', 'app/models/spm_en_hi.model')
print("Models copied to app/models/")

Models copied to app/models/
